# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their IDs. Each record set, field, and column is referenced by its unique `@id` identifier.

In [ ]:
# List all available record sets with their @id and name.
record_sets = dataset.metadata.record_sets

print(f"Record sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"@id: {rs.id}\n  name: {rs.name}\n  description: {rs.description}\n")

# Let's choose the most comprehensive record set for demonstration, usually the main record set.
if record_sets:
    primary_record_set = record_sets[0]
    print(f"Selected main record set for exploration: {primary_record_set.name} (@id={primary_record_set.id})")

    fields = primary_record_set.fields
    print("\nFields in this record set:")
    for field in fields:
        print(f"  @id: {field.id}\n    name: {field.name}\n    description: {field.description}\n    dataType: {field.data_type}\n")
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# List all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display the available columns for the primary record set
main_record_set_id = record_set_ids[0]
print(f"Fields in DataFrame for {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping by key attributes. Reference fields by their `@id`.

Modify `numeric_field_id` and `group_field_id` as appropriate based on the field list above.

In [ ]:
# Select a numeric field for analysis by @id. Replace with actual @id from your dataset fields overview.
numeric_field_id = None
group_field_id = None
# Automatically try to pick a numeric field (usually float/integer fields)
fields = dataset.metadata.record_sets[0].fields
for field in fields:
    if field.data_type in ["Float", "Integer", "Number"] and numeric_field_id is None:
        numeric_field_id = field.id
    # Try to pick a second field for demonstration (likely a categorical one)
    if group_field_id is None and field.data_type in ["Text", "String"]:
        group_field_id = field.id

if numeric_field_id is None:
    raise RuntimeError("No numeric field found for analysis. Please set 'numeric_field_id' explicitly.")

print(f"Using numeric field: {numeric_field_id}")
if group_field_id:
    print(f"Using group-by field: {group_field_id}")

df = dataframes[main_record_set_id]

# Remove missing or non-numeric entries
df = df.copy()
df_num = pd.to_numeric(df[numeric_field_id], errors='coerce')
df = df[df_num.notnull()]
df[numeric_field_id] = df_num[df_num.notnull()]

threshold = df[numeric_field_id].quantile(0.75)  # Use 75th percentile as threshold for illustration
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by another field if available
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Plot distribution of the numeric field
plt.figure(figsize=(8, 4))
plt.hist(df[numeric_field_id], bins=30, alpha=0.7)
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# If grouped data available, plot mean value per group
if 'grouped_df' in locals():
    plt.figure(figsize=(10, 4))
    grouped_df.plot(kind='bar', x=group_field_id, y=numeric_field_id, ax=plt.gca(), legend=False)
    plt.ylabel(f"Mean of {numeric_field_id}")
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to:
- Load and explore Croissant-structured datasets,
- Inspect record sets and fields by their `@id`,
- Load data into pandas DataFrames,
- Apply basic EDA, filtering, normalization, and grouping,
- Visualize key data characteristics.

Adjust field IDs and parameters as needed to suit your specific data exploration and analysis goals.